# Q-Rutas Piura · demo de hackatón

Ruteo híbrido clásico-cuántico de ayuda humanitaria para las comunidades que el desborde del río Piura deja aisladas.

Este cuaderno recorre el flujo paso a paso: escenario, fase clásica (triage y clustering), QUBO de un cluster, QAOA en sus dos variantes, comparación con métodos clásicos sobre el mismo QUBO y, al final, el pipeline completo con mapas y métricas. El modelo matemático está explicado en el `README.md`.

In [ ]:
# En Google Colab: sube qrutas_piura.zip al entorno y descomenta estas líneas
# !unzip -q qrutas_piura.zip
# %cd qrutas_piura
# !pip install -q -r requirements.txt

In [ ]:
import os
import sys

RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, RAIZ)
os.chdir(RAIZ)

import numpy as np
from qrutas.config import Config
from qrutas.datos import cargar_escenario
from qrutas.costos import construir_instancia
from qrutas.clustering import clustering_instancia, metricas_reduccion
from qrutas import qubo as qb, cuantico, clasicos

print("Qiskit instalado:", cuantico.qiskit_disponible())

## 1. Escenario

Comunidades con coordenadas del padrón de centros poblados del INEI. El estado de los puentes y de los accesos terrestres es el del escenario (se cambia en `datos/`).

In [ ]:
cfg = Config()
esc = cargar_escenario(cfg)
inst = construir_instancia(esc, cfg)

MODO = {"camion": "camión", "helicoptero": "helicóptero"}
print(f"{'comunidad':28s} {'distrito':12s} {'fam':>4s} {'kg':>6s}  prioridad  acceso     modo         ventana")
for n in esc.comunidades:
    i = n.idx
    print(f"{n.nombre:28s} {n.distrito_nombre:12s} {n.familias:4d} {inst.demanda[i]:6.0f}  "
          f"{'sí' if n.prioridad_medica else 'no':9s}  {n.acceso_terrestre:9s}  {MODO[inst.tipo_por_nodo[i]]:12s} {inst.ventana[i]:5.1f} h")
print()
for p in esc.puentes:
    print(f"{p.id}  {p.nombre:45s} {p.estado}")

## 2. Fase clásica: triage modal y clustering

Cada cluster es un viaje (capacidad del vehículo y máximo 4 paradas, porque 4 paradas son 16 qubits). La asignación a medoides se resuelve como problema entero y luego se refina con búsqueda local.

In [ ]:
clusters, info = clustering_instancia(inst)
red = metricas_reduccion(inst, clusters)
print(f"Sin clustering, el VRP como QUBO pediría {red['qubits_vrp_completo']} qubits.")
print(f"Con clustering: {red['n_subproblemas']} subproblemas, qubits por subproblema: {red['qubits_por_subproblema']}")
print(f"Refinamiento: costo estimado {info['J_proxy_inicial']:.2f} -> {info['J_proxy_final']:.2f}\n")
for c in clusters:
    print(f"{c.id:8s} {MODO[c.tipo]:12s} {inst.demanda[c.nodos].sum():6.0f} kg  ", ", ".join(esc.nodos[i].nombre for i in c.nodos))

## 3. El QUBO de un cluster

Tomamos el cluster más grande (16 qubits). Comprobamos dos cosas: que la energía del QUBO de cada ruta coincide (o es cota inferior, por las ventanas de medicinas) con su costo exacto, y que el mínimo global entre los $2^{16}$ estados es una ruta válida.

In [ ]:
c = max(clusters, key=lambda c: (len(c.nodos), c.id))
sp = qb.construir_subproblema(inst, c, inicio_h=0.0)
print(f"{sp.id}: {sp.n} paradas, {sp.nq} qubits, penalización A = {sp.A:.3f}")

import itertools
for pm in itertools.permutations(range(sp.n)):
    e = qb.energia(sp.Q, sp.const, qb.perm_a_x(pm, sp.n))
    j = qb.costo_exacto(sp, pm)
    ruta = " -> ".join(esc.nodos[sp.nodos[i]].nombre for i in pm)
    print(f"E = {e:8.3f}   costo exacto = {j:8.3f}   {ruta}")

E = qb.energias_todas(sp.Q, sp.const)
k = int(np.argmin(E))
print("\nMínimo global entre", len(E), "estados es ruta válida:", qb.x_a_perm(qb.indice_a_x(k, sp.nq), sp.n) is not None)
h, J, off = qb.a_ising(sp.Q, sp.const)
print("Términos Ising: Z =", int(np.sum(np.abs(h) > 1e-12)), "· ZZ =", int(np.sum(np.abs(J) > 1e-12)))

## 4. QAOA: estándar y con mezclador que preserva rutas (XY)

Entrenamiento con CVaR al 15 %, rejilla en p = 1, COBYLA y crecimiento capa a capa (INTERP). Al final se muestrean 4096 disparos y se toma la mejor ruta válida medida.

In [ ]:
prep = cuantico.preparar(sp)
cfg_demo = Config(p_max=2, maxiter_cobyla=150)
rng = np.random.default_rng(7)
for variante in ("estandar", "xy"):
    res, muestras, ir = cuantico.ejecutar_qaoa(sp, prep, cfg_demo, variante, rng, motor="numpy")
    p_opt = res.probs[prep.optimos].sum()
    p_fac = res.probs[prep.factible].sum()
    unicos = np.unique(muestras)
    validos = unicos[prep.factible[unicos]]
    mejor = validos[np.argmin(prep.E_total[validos])] if len(validos) else None
    ruta = qb.x_a_perm(qb.indice_a_x(int(mejor), sp.nq), sp.n) if mejor is not None else None
    print(f"QAOA {variante:8s} p={res.p}  P(ruta válida)={p_fac:.3f}  P(óptimo)={p_opt:.4f}  "
          f"evaluaciones={res.n_evaluaciones}  tiempo={res.tiempo_s:.1f} s")
    print("   mejor ruta medida:", " -> ".join(esc.nodos[sp.nodos[i]].nombre for i in ruta) if ruta else "ninguna válida")
    print(f"   brecha al óptimo: {100 * (prep.E_total[mejor] - prep.E_opt) / abs(prep.E_opt):.2f} %\n")

In [ ]:
# El mismo circuito en Qiskit (si está instalado)
if cuantico.qiskit_disponible():
    res, muestras, ir = cuantico.ejecutar_qaoa(sp, prep, cfg_demo, "estandar", rng, motor="auto")
    qc = cuantico.ir_a_qiskit(ir, res.gammas, res.betas, medir=True)
    print(qc.name, "| qubits:", qc.num_qubits, "| profundidad:", qc.depth(), "| compuertas:", dict(qc.count_ops()))
    print("Muestreo hecho con:", res.backend_muestreo)
    from qrutas import hardware
    t = hardware.transpilar_ibm(ir, res.gammas, res.betas)
    if t:
        print("Transpilado a", t["backend"], "->", t["compuertas_2q"], "compuertas de 2 qubits, profundidad", t["profundidad"])
else:
    print("Qiskit no está instalado: pip install qiskit qiskit-aer qiskit-ibm-runtime")

## 5. El mismo QUBO con métodos clásicos

In [ ]:
sa = clasicos.sa_qubo(sp, prep, cfg, rng)
ga = clasicos.ga_permutaciones(sp, prep, cfg, rng)
fb = clasicos.fuerza_bruta(sp)
print(f"Simulated Annealing: P(óptimo) por lectura = {sa['p_optimo']:.2f}, tiempo = {sa['tiempo_s'] * 1000:.0f} ms")
print(f"Algoritmo genético:  brecha = {100 * (ga['E'] - prep.E_opt) / abs(prep.E_opt):.2f} %, tiempo = {ga['tiempo_s'] * 1000:.0f} ms")
print(f"Fuerza bruta:        {fb['n_evaluaciones']} rutas en {fb['tiempo_s'] * 1000:.1f} ms")

## 6. Pipeline completo

Todos los clusters con QAOA, SA, GA y fuerza bruta, más el clásico puro sin clustering. Tarda alrededor de un minuto.

In [ ]:
from qrutas.pipeline import ejecutar, guardar
from qrutas.visualizacion import generar_figuras
from qrutas.reporte import escribir_resumen

R = ejecutar(Config())
guardar(R, "resultados")
figuras = generar_figuras(R, "resultados")
escribir_resumen(R, figuras, "resultados/resumen.md")

In [ ]:
try:
    from IPython.display import Image, Markdown, display
    display(Markdown(open("resultados/resumen.md", encoding="utf-8").read()))
    for ruta in figuras.values():
        display(Image(filename=ruta, width=1100))
except ImportError:
    print(open("resultados/resumen.md", encoding="utf-8").read())

## 7. Para el pitch

- La fase clásica convierte un problema de 784 qubits en subproblemas de 16 o menos: sin eso no hay nada que correr en un QPU actual.
- El QAOA encuentra la ruta óptima de cada viaje, pero por post-selección. El mezclador que preserva rutas (XY) multiplica la probabilidad de acertar.
- A esta escala no hay ventaja cuántica: SA lo hace en milisegundos. Lo medimos y lo decimos.
- La diferencia con el mejor clásico viene de la descomposición (máximo 4 paradas por viaje), no del QAOA.